# Projet ML for Econometrics

## Introduction

Ce notebook retrace l'ensemble de notre travail effectué dans le cadre du projet pour le cours de ML for Econometrics présenté par Mr. Doutreligne et Mr. Crépon

## Présentation de la base de données et de la problématique

Nous nous appuyons sur le dataset "Predict Students' Dropout and Academic Success" qui est une base de données disponible publiquement via le lien suivant https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

Cette base de données répertorie le parcours scolaire de 4424 étudiants dans différents cursus (agronomie, design, infirmier, etc.). La BDD donne pour chaque individu de nombreuses informations socio-démographiques de l'élève (sexe, âge, niveau scolaire des deux parents, nationalité, etc.) ainsi que des données sur son parcours scolaire (note durant les deux semestres de l'année) ainsi que la situation de l'élève à la fin du temps scolaire.

Cette dernière variable est ici notre variable d'intérêt, elle peut prendre trois valeurs différentes : dropout si l'élève a abandonné les études, graduate si l'élève a été diplômé, ou enrolled si l'élève continue les cours dans un temps supplémentaire.

Notre travail se concentre sur la recherche de l'effet causal d'avoir une bourse d'études sur la réussite de l'élève.

## Formulation PICO de notre problématique

P : La population étudiée ici est un ensemble d'étudiants étant en études entre 2008 et 2018

I : L'intervention que l'on considère ici est le fait de recevoir une bourse durant ses études

C : On compare cela avec les étudiants n'ayant pas reçu de bourse durant leur études

O : L'outcome considéré est le fait d'obtenir son diplôme à la fin du parcours scolaire

## Introduction, analyse sociologique et DAG

## Analyse descriptive des données

L'objectif de cette seconde partie est de visualiser en détails la base de données, en particulier on s'attardera particulièrement sur l'outcome et sur les variables corrélées avec le fait d'avoir une bourse

### Téléchargement des données et nettoyage de la BDD

In [3]:
#pip install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo 


In [ ]:

print("--- 1. Chargement du dataset ---")
dataset = fetch_ucirepo(id=697) 
X = dataset.data.features 
y = dataset.data.targets 
X["Application mode"] = X["Application mode"].astype(str)
X["Course"] = X["Course"].astype(str)
X["Father\'s occupation"] = X["Father\'s occupation"].astype(str)
X["Mother\'s occupation"] = X["Mother\'s occupation"].astype(str)
X["Father\'s qualification"] = X["Father\'s qualification"].astype(str)
X["Mother\'s qualification"] = X["Mother\'s qualification"].astype(str)
X["Marital Status"] = X["Marital Status"].astype(str)
X["Nacionality"] = X["Nacionality"].astype(str)
X["Previous qualification"] = X["Previous qualification"].astype(str)
df = pd.concat([X, y], axis=1)

print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes\n")

print("--- 2. Types de variables et Valeurs manquantes ---")
valeurs_manquantes = pd.DataFrame({
    'Type': df.dtypes,
    'Manquants (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Valeurs Uniques': df.nunique()
})
print(valeurs_manquantes)

print("\n--- 3. Statistiques Descriptives ---")
display(df.describe().T.round(2))

--- 1. Chargement du dataset ---
Dimensions du dataset : 4424 lignes, 37 colonnes

--- 2. Types de variables et Valeurs manquantes ---
                                                   Type  Manquants (%)  \
Marital Status                                   object            0.0   
Application mode                                 object            0.0   
Application order                                 int64            0.0   
Course                                           object            0.0   
Daytime/evening attendance                        int64            0.0   
Previous qualification                           object            0.0   
Previous qualification (grade)                  float64            0.0   
Nacionality                                      object            0.0   
Mother's qualification                           object            0.0   
Father's qualification                           object            0.0   
Mother's occupation                              ob

/tmp/ipykernel_11669/280577157.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Application mode"] = X["Application mode"].astype(str)
/tmp/ipykernel_11669/280577157.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Course"] = X["Course"].astype(str)
/tmp/ipykernel_11669/280577157.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/

,count,mean,std,min,25%,50%,75%,max
Application order,4424.0,1.73,1.31,0.00,1.00,1.00,2.00,9.00
Daytime/evening attendance,4424.0,0.89,0.31,0.00,1.00,1.00,1.00,1.00
Previous qualification (grade),4424.0,132.61,13.19,95.00,125.00,133.10,140.00,190.00
Admission grade,4424.0,126.98,14.48,95.00,117.90,126.10,134.80,190.00
Displaced,4424.0,0.55,0.50,0.00,0.00,1.00,1.00,1.00
Educational special needs,4424.0,0.01,0.11,0.00,0.00,0.00,0.00,1.00
Debtor,4424.0,0.11,0.32,0.00,0.00,0.00,0.00,1.00
Tuition fees up to date,4424.0,0.88,0.32,0.00,1.00,1.00,1.00,1.00
Gender,4424.0,0.35,0.48,0.00,0.00,0.00,1.00,1.00
Scholarship holder,4424.0,0.25,0.43,0.00,0.00,0.00,0.00,1.00


Comme nous le montrent ces résultats, la base de données ne contient pas de valeur manquante.

Cependant, ce résultat cache d'une certaine façon la réalité : Rappelons que dans note BDD, certains élèves abandonnent leurs études au cours de l'année.
Ainsi, les variables inquant le nombre d'ECTS crédités par semestre ainsi que les notes des élèves n'a pas trop de signification : en effet, un élève qui abandonne son année aura administrativement des notes nulles pour le reste de l'année.

Comme l'idée de notre projet est d'étudier les facteurs qui influent en amont sur le fait d'avoir une bourse et de réussir son année, ces variables en question ne semblent pas pertinentes et risquent de nous induire en erreur. Ainsi, on les supprime du dataset

Notons également que certaines variables, comme le travail des parents ou la nationalité de l'élève, sont des variables nominales encodées en entier. Il est donc important de les transformer en string pour ne pas faire d'erreur par la suite.

In [8]:
data_propre = df = df.drop([

    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
    
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)"

], axis=1)

In [10]:
data_propre.describe(include="all")

,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Unemployment rate,Inflation rate,GDP,Target
count,4424,4424,4424.000000,4424,4424.000000,4424,4424.000000,4424,4424,4424,...,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424
unique,6,18,NaN,17,NaN,17,NaN,21,29,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,1,1,NaN,9500,NaN,1,NaN,1,1,37,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Graduate
freq,3919,1708,NaN,766,NaN,3717,NaN,4314,1069,1209,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2209
mean,NaN,NaN,1.727848,NaN,0.890823,NaN,132.613314,NaN,NaN,NaN,...,0.113698,0.880651,0.351718,0.248418,23.265145,0.024864,11.566139,1.228029,0.001969,NaN
std,NaN,NaN,1.313793,NaN,0.311897,NaN,13.188332,NaN,NaN,NaN,...,0.317480,0.324235,0.477560,0.432144,7.587816,0.155729,2.663850,1.382711,2.269935,NaN
min,NaN,NaN,0.000000,NaN,0.000000,NaN,95.000000,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.000000,17.000000,0.000000,7.600000,-0.800000,-4.060000,NaN
25%,NaN,NaN,1.000000,NaN,1.000000,NaN,125.000000,NaN,NaN,NaN,...,0.000000,1.000000,0.000000,0.000000,19.000000,0.000000,9.400000,0.300000,-1.700000,NaN
50%,NaN,NaN,1.000000,NaN,1.000000,NaN,133.100000,NaN,NaN,NaN,...,0.000000,1.000000,0.000000,0.000000,20.000000,0.000000,11.100000,1.400000,0.320000,NaN
75%,NaN,NaN,2.000000,NaN,1.000000,NaN,140.000000,NaN,NaN,NaN,...,0.000000,1.000000,1.000000,0.000000,25.000000,0.000000,13.900000,2.600000,1.790000,NaN


### Statistiques Descriptives